#  NFPP Sodium-Ion Energy Storage Evaluation for Distribution Networks

This notebook implements the complete research pipeline for DFN-based BESS optimization 

In [ ]:
import os
import sys

# Kaggle Environment Setup  
REPO_DIR = "/kaggle/working/sodium-ion-ess"  
REPO_URL = "https://github.com/mhizterpaul/sodium-ion-ess.git"  

if not os.path.exists(REPO_DIR):  
    subprocess.run( ["git", "clone", REPO_URL, REPO_DIR], check=True, )  
os.chdir(REPO_DIR)  
if REPO_DIR not in sys.path:  
    sys.path.insert(0, REPO_DIR) 

print(f"Current working directory: {os.getcwd()}") 
 
# MP API Key configuration 
MP_API_KEY='' 
 
!pip install pybamm numpy scipy requests mp-api pymatgen pymoo mpi4py ufl 
 
import pybamm 
import numpy as np 
import scipy 
import pywt 

print("Environment initialized successfully.")

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2: Running Hierarchical Material & Structural Optimization...")
optimizer = HierarchicalOptimizer()
optimized_res = optimizer.run()

print("\n--- OPTIMIZATION RESULTS ---")
print("Optimized Design Variables per Objective:")
for obj, specs in optimized_res.get("opt_designs_per_objective", {}).items():
    print(f"\nObjective: {obj.capitalize()}")
    for k, v in specs.items():
        print(f"  {k:40s}: {v:12.6e}")

print("\nSelected Integrated Design Variables:")
for k, v in optimized_res.get("design_specs_representative", {}).items():
    print(f"  {k:40s}: {v:12.6e}")

print("\n--- OPTIMAL CANDIDATE: QM DATA & DERIVED CELL PARAMETERS ---")
mats = optimized_res.get("materials", {})
deltas = optimized_res.get("combined_deltas_representative", {})
for cat in ["cathode", "electrolyte"]:
    print(f"\n{cat.capitalize()} Material:")
    m_data = mats.get(cat, {})
    print(f"  Name: {m_data.get('name') or m_data.get('salt')}")
    print(f"  Formula: {m_data.get('formula')}")
    print("  QM/Physics Properties:")
    for pk, pv in m_data.get("properties", {}).items():
        print(f"    {pk:25s}: {pv}")

print("\nMapping to PyBaMM Parameter Deltas:")
for category, props in deltas.items():
    print(f"  [{category.upper()}]")
    for pk, pv in props.items():
        print(f"    {pk:45s}: {pv:+.4e}")

## Stage 3: Stability Validation & Parameter Extraction
Performance evaluation and resistance profile generation for the digital twin.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=optimizer.engine)
results = validator.run_validation()

print("\nStage 3.1: Running BESS Robustness Evaluation...")
from src.simulation.tests import BESSEvaluator
bess_evaluator = BESSEvaluator(optimized_res)
envelope_res = bess_evaluator.evaluate_bess_performance()

import pandas as pd
from IPython.display import display, HTML

metrics_meta = [
    ("round_trip_energy_efficiency", "Round-Trip Energy Efficiency (RTE)", "eta_RTE", "{:.2%}"),
    ("coulombic_efficiency", "Coulombic Efficiency", "eta_C", "{:.2%}"),
    ("voltage_efficiency", "Voltage Efficiency", "eta_V", "{:.2%}"),
    ("usable_energy_capacity_wh", "Usable Energy Capacity", "E_usable", "{:.2f} Wh"),
    ("power_capability_w", "Power Capability", "P_max", "{:.2f} W"),
    ("thermal_response_delta_t", "Thermal Response Delta T", "Delta T", "{:.2f} K"),
    ("max_temperature_k", "Maximum Temperature", "T_max", "{:.2f} K"),
    ("depth_of_discharge", "Depth of Discharge", "DoD", "{:.2%}"),
    ("equivalent_full_cycles", "Equivalent Full Cycles", "EFC", "{:.4f}"),
    ("capacity_fade", "Capacity Fade", "F_Q", "{:.4e}"),
    ("cycle_life", "Estimated Cycle Life", "N_life", "{:.0f} cycles"),
    ("calendar_life_years", "Estimated Calendar Life", "t_life", "{:.1f} years"),
    ("levelized_cost_of_storage_usd_per_kwh", "Levelized Cost of Storage", "LCOS", "${:.4f}/kWh")
]

rows = []
for key, desc, sym, fmt in metrics_meta:
    val = envelope_res.get(key, 0.0)
    rows.append({"Metric": desc, "Symbol": sym, "Value": fmt.format(val)})

df_metrics = pd.DataFrame(rows)
display(HTML("<h3>NFPP BESS Robustness Evaluation Framework Metrics (paper.md aligned)</h3>"))
display(df_metrics)